# Papapanagiotou Panagiotis

## **Άσκηση – Prompt Evaluation**
### Δίνονται τα παρακάτω prompts:
- ####  **Prompt A (κακό prompt)**
   - ####  “Explain APIs”
- ####  **Prompt B (μέτριο prompt)**
   - ####  “Explain what an API is and give an example”
- ####  **Prompt C (καλό prompt)**
   - ####  “You are a software architect. Explain what an API is to non-technical business stakeholders. Use a real-world analogy and provide one practical example from web development. Structure your answer in bullet points.”

- ### Για κάθε prompt, αξιολογήστε με βάση:
    1. Σαφήνεια (clarity) (1–5)
    2. Context (1–5)
    3. Καθορισμός ρόλου (persona) (1–5)
    4. Ποιότητα αναμενόμενης εξόδου (1–5)
    5. Format (1–5)
- ### Δώστε βαθμολογία και αιτιολόγηση.

# Λύση άσκησης
#### Θα χρησιμοποιήσω την Antropic ώς αξιολογιτή και την OpenAi ώς αξιολογούμενο.
### My basic setup code


In [11]:
# ── Setup ──────────────────────────────────────────────
import os
from pathlib import Path
from dotenv import load_dotenv
from IPython.display import display, Markdown, clear_output
 
load_dotenv(Path.cwd().parent / "../module_02_prompt_enginnering/.env")
 
from openai import OpenAI
 
client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))
 
OPENAI_MODEL = "gpt-4o-mini" 
if client:
    print(f"OpenAI client ready - using model {OPENAI_MODEL}")

OpenAI client ready - using model gpt-4o-mini


In [3]:
from anthropic import Anthropic

claude_client = Anthropic(api_key=os.getenv("ANTHROPIC_API_KEY"))
CLAUDE_MODEL = "claude-sonnet-4-6" 

if claude_client:
    print(f"Antropic client ready - using model {CLAUDE_MODEL}")

Antropic client ready - using model claude-sonnet-4-6


In [12]:
def ask_anthropic(prompt: str, system: str | None = None, temperature: float= 0.7, max_resp_tokens: int= 800, ai_model: str= "claude-sonnet-4-6", output_format: str | None = None) -> None:
    """
    Send a prompt to Anthropic using the streaming Messages API and return
    the complete generated text.

    Args:
        prompt (str):
            The user prompt to send to the model.
        system (str | None, optional):
            Optional system instruction for the model. In the streaming API
            for this SDK version, it is converted into the required list format.
            Defaults to None.
        temperature (float, optional):
            Controls randomness of the response. Defaults to 0.7.
        max_resp_tokens (int, optional):
            Maximum number of tokens in the generated response. Defaults to 800.
        ai_model (str, optional):
            The model name to use. Defaults to "claude-sonnet-4-6".
        output_format (str | None, optional):
            Optional output format for the streaming API, if supported by the
            installed SDK version. Defaults to None.

    Returns: None
    """

    params = {
        "model": ai_model,
        "max_tokens" : max_resp_tokens,
        "temperature": temperature,
        "messages": [
            {'role': 'user', 'content': prompt}
        ],
    }

    if system:
        params["system"] = [{"type": "text", "text": system}]

    if output_format:
        params["output_format"] = output_format
    
    with claude_client.messages.stream(**params) as stream:
        full_text = ""
        for text in stream.text_stream:
            full_text += text
            clear_output(wait=True)
            #display(Markdown(full_text))
    
    return full_text

display(Markdown(f"### **The function has run**"))

### **The function has run**

In [6]:
def ask_open_ai(prompt: str, system_content: str= None, temperature: float= 0.7, max_resp_tokens: int= 800, ai_model:  str="gpt-4o-mini") -> str:
    """
    Send a prompt to the OpenAI Chat Completions API and return the assistant's reply.

    This helper function builds a chat message list, optionally including a system
    instruction, sends it to the specified model, and returns the generated text
    content from the first response choice.

    Args:
        prompt (str): The user prompt to send to the model.
        system_content (str, optional): Optional system-level instruction that
            defines the assistant's behavior, tone, or constraints. If provided,
            it is added as the first message in the conversation. Defaults to None.
        temperature (float, optional): Sampling temperature used to control
            randomness in the model's response. Lower values produce more
            deterministic outputs; higher values produce more creative or varied
            outputs. Defaults to 0.7.
        max_resp_tokens (int, optional): Maximum number of tokens allowed in the
            generated response. Defaults to 800.
        ai_model (str, optional): The model name to use for the request.
            Defaults to "gpt-4o-mini".

    Returns:
        str: The text content of the assistant's first response message.

    Raises:
        Exception: Propagates any exception raised by the OpenAI client request
            (for example, authentication errors, rate limits, invalid parameters,
            or network-related failures).

    Example of use:
         reply = ask("Explain recursion in simple terms.")
         print(reply)

        reply = ask(
             prompt="Write a haiku about Python.",
             system_content="You are a poetic assistant.",
             temperature=0.9,
             max_resp_tokens=100,
             ai_model="gpt-4o-mini"
        )        
        print(reply)
    """
    msgs = []
    if system_content:
        msgs.append({"role": "system", "content": system_content})
    msgs.append({"role":"user", "content": prompt})
    resp = client.chat.completions.create(
        model=ai_model,
        messages=msgs,
        temperature=temperature,
        max_tokens=max_resp_tokens
    )
    return resp.choices[0].message.content

display(Markdown(f"### **The function has run**"))

### **The function has run**

## Παρακάτω βάζω τα prompts, παίρνω τις απαντήσεις και βάζω τα κριτήρια. 

In [8]:
prompt_a = "Explain APIs"
prompt_b = "Explain what an API is and give an example"
prompt_c = "You are a software architect. Explain what an API is to non-technical business stakeholders. Use a real-world analogy and provide one practical example from web development. Structure your answer in bullet points."

response_a = ask_open_ai(prompt_a)
response_b = ask_open_ai(prompt_b, max_resp_tokens= 1000)
response_c = ask_open_ai(prompt_c, max_resp_tokens= 1200)

criteria = (
    "Clarity (0-5)." 
    "Context (0-5)." 
    "Persona (0-5)." 
    "Output quality (0-5)."
    "Format (0-5)."
    "Score each dimension, then average for the final score."
)

display(Markdown(f"### **The variables created and loaded**"))

### **The variables created and loaded**

## Τροποποιώ την συνάρτηση αξιολόγησης που κάναμε στο μάθημα.

In [13]:
def evaluate_response(question: str, response: str, criteria: str) -> dict:  
    """Use the Antropic LLM to evaluate this AI response on a scale of 0-5"""
    import json
    
    eval_system = (
        "You are an expert prompt-engineering evaluator.\n"
        "You will receive a QUESTION, a RESPONSE, and CRITERIA.\n\n"
        "Score EACH dimension 0-5, then compute the average as 'score'.\n\n"
        "Return ONLY valid JSON (no markdown fences). Schema:\n"
        '{"score": <float>, "breakdown": {"clarity": <int>, "context": <int>, '
        '"persona": <int>, "quality": <int>, "format": <int>}, "one_liner": "<= 15 word summary"}'
    )

    eval_prompt = (
        f"QUESTION:\n{question}\n\n"
        f"RESPONSE:\n{response}\n\n"
        f"CRITERIA:\n{criteria}"
    )
    
    raw = ask_anthropic(prompt=eval_prompt, system= eval_system, temperature=0.0)

    try:
        cleaned = raw.strip()
        if cleaned.startswith("```json"):
            cleaned = cleaned.removeprefix("```json").removesuffix("```").strip()
        elif cleaned.startswith("```"):
            cleaned = cleaned.removeprefix("```").removesuffix("```").strip()
 
        return json.loads(cleaned)
    except json.JSONDecodeError:
        return {"score": 0, "breakdown": {}, "one_liner": "⚠ JSON parse error"}

display(Markdown("### The function has run"))

### The function has run

## Παρακάτω κάνω τις αξιολογίσεις 

In [14]:
eval_a = evaluate_response(prompt_a, response_a, criteria)
eval_b = evaluate_response(prompt_b, response_b, criteria)
eval_c = evaluate_response(prompt_c, response_c, criteria)

display(Markdown("### **The evaluation variables has load**"))

### **The evaluation variables has load**

## Προετοιμάζω και εμφάνιζω τα αποτελέσματα της αξιολόγησης.

In [20]:
scores = {"A": eval_a.get('score', 0), "B": eval_b.get('score', 0), "C": eval_c.get('score', 0)}
winner = max(scores)

md = f"""
# Test Results
 
## Prompts:
 - Prompt A:  **{prompt_a}**
 - Prompt B:  **{prompt_b}**
 - Prompt C:  **{prompt_c}**

 
| Metric | Prompt A (Bad) | Prompt B (mediocre) | Prompt C (Good prompt) |
|---|---:|---:|---:|
| **Overall Score** | {eval_a.get('score', 'N/A')}/5 | {eval_b.get('score', 'N/A')}/5 | {eval_c.get('score', 'N/A')}/5 |
| **Clarity** | {eval_a.get('breakdown', {}).get('clarity', 'N/A')}/5 | {eval_b.get('breakdown', {}).get('clarity', 'N/A')}/5 | {eval_c.get('breakdown', {}).get('clarity', 'N/A')}/5 |
| **Context** | {eval_a.get('breakdown', {}).get('context', 'N/A')}/5 | {eval_b.get('breakdown', {}).get('context', 'N/A')}/5 | {eval_c.get('breakdown', {}).get('context', 'N/A')}/5 |
| **Persona** | {eval_a.get('breakdown', {}).get('persona', 'N/A')}/5 | {eval_b.get('breakdown', {}).get('persona', 'N/A')}/5 | {eval_c.get('breakdown', {}).get('persona', 'N/A')}/5 |
| **Quality** | {eval_a.get('breakdown', {}).get('quality', 'N/A')}/5 | {eval_b.get('breakdown', {}).get('quality', 'N/A')}/5 | {eval_c.get('breakdown', {}).get('quality', 'N/A')}/5 |
| **Format** | {eval_a.get('breakdown', {}).get('format', 'N/A')}/5 | {eval_b.get('breakdown', {}).get('format', 'N/A')}/5 | {eval_c.get('breakdown', {}).get('format', 'N/A')}/5 | 

 
## Prompt A Response
{response_a}
 
### **Judge summary:** {eval_a.get('one_liner', 'N/A')}
 
---
 
## Prompt B Response
{response_b}
 
### **Judge summary:** {eval_b.get('one_liner', 'N/A')}
 
---

## Prompt C Response
{response_c}
 
### **Judge summary:** {eval_c.get('one_liner', 'N/A')}
 
---
 
# 🏆 Winner: Prompt {winner}
"""
 
display(Markdown(md))



# Test Results

## Prompts:
 - Prompt A:  **Explain APIs**
 - Prompt B:  **Explain what an API is and give an example**
 - Prompt C:  **You are a software architect. Explain what an API is to non-technical business stakeholders. Use a real-world analogy and provide one practical example from web development. Structure your answer in bullet points.**


| Metric | Prompt A (Bad) | Prompt B (mediocre) | Prompt C (Good prompt) |
|---|---:|---:|---:|
| **Overall Score** | 4.2/5 | 4.4/5 | 4.4/5 |
| **Clarity** | 5/5 | 5/5 | 5/5 |
| **Context** | 4/5 | 4/5 | 4/5 |
| **Persona** | 3/5 | 4/5 | 4/5 |
| **Quality** | 5/5 | 5/5 | 5/5 |
| **Format** | 4/5 | 4/5 | 4/5 | 


## Prompt A Response
API stands for Application Programming Interface, which is a set of rules and protocols that allows different software applications to communicate with each other. APIs enable the interaction between different software components, allowing them to share data and functionalities in a standardized way.

### Key Concepts of APIs:

1. **Endpoints**: These are specific paths or URLs through which an API can be accessed. Each endpoint represents a specific function or resource.

2. **Requests and Responses**: When a client (like a web application or mobile app) wants to access data or functionality from a server via an API, it sends a request. The server processes this request and sends back a response, typically in a format like JSON or XML.

3. **HTTP Methods**: APIs often use standard HTTP methods to perform operations:
   - **GET**: Retrieve data from the server.
   - **POST**: Send data to the server to create a new resource.
   - **PUT**: Update an existing resource on the server.
   - **DELETE**: Remove a resource from the server.

4. **Authentication**: Many APIs require authentication to ensure that only authorized users can access certain features or data. Common methods include API keys, OAuth tokens, and basic authentication.

5. **Rate Limiting**: To prevent abuse, APIs may impose limits on how many requests a client can make in a certain timeframe.

6. **Versioning**: APIs may be versioned to allow developers to introduce changes or improvements without breaking existing functionality for users.

### Types of APIs:

1. **Web APIs**: These are APIs accessible via the web using HTTP/HTTPS. They are widely used for web services and applications.

2. **REST APIs**: Representational State Transfer (REST) is an architectural style for designing networked applications. REST APIs use standard HTTP methods and are stateless.

3. **SOAP APIs**: Simple Object Access Protocol (SOAP) is a protocol for exchanging structured information over the web. SOAP APIs are more rigid than REST and use XML for messaging.

4. **GraphQL APIs**: A query language for APIs that allows clients to request exactly the data they need, rather than receiving a fixed structure, as is common with REST.

5. **Library APIs**: These are APIs provided by programming libraries or frameworks that allow developers to utilize pre-written code functionalities in their applications.

### Use Cases:

- **Social Media Integration**: Allowing applications to access social media data (e.g., posting updates, retrieving user profiles).
- **Payment Processing**: Facilitating online transactions through services like PayPal or Stripe.
- **Data Retrieval**: Accessing external databases or services (e.g., weather data, stock prices).
- **Microservices Communication**: Enabling different parts of an application to interact with each other in a microservices architecture.

In summary, APIs are essential for modern software development, enabling seamless communication between different systems and services, enhancing functionality, and fostering innovation by allowing developers to build on existing technologies.

### **Judge summary:** Comprehensive, well-structured API explanation lacking audience-tailored persona adaptation

---

## Prompt B Response
An API, or Application Programming Interface, is a set of rules and protocols that allows different software applications to communicate with each other. It defines the methods and data formats that applications can use to request and exchange information. APIs enable developers to access functionalities or data from another service without needing to understand the internal workings of that service.

### Key Features of APIs:
1. **Communication**: APIs facilitate communication between different software systems.
2. **Abstraction**: They hide the complex implementation details, allowing developers to use functionalities without needing to know how they are implemented.
3. **Interoperability**: APIs allow different systems, built on diverse technologies, to work together.

### Example of an API:
A common example of an API is the **Twitter API**. This API allows developers to interact with Twitter's platform programmatically. For instance, a developer can use the Twitter API to:
- Retrieve tweets from a user's timeline.
- Post a new tweet.
- Search for tweets containing specific keywords.

In this case, developers send HTTP requests to the Twitter API endpoints, and the API responds with the requested data in a structured format, usually JSON. This enables the integration of Twitter functionalities into other applications, such as social media dashboards or analytics tools.

### **Judge summary:** Clear, well-structured API explanation with a relevant real-world example.

---

## Prompt C Response
Certainly! Here’s an explanation of what an API (Application Programming Interface) is, tailored for non-technical business stakeholders, using a real-world analogy and a practical example from web development.

### Explanation of API

- **Definition**: An API is a set of rules and protocols that allows different software applications to communicate with each other. It acts as a bridge that enables different systems to work together.

### Real-World Analogy: Restaurant Menu

- **Restaurant Menu**: Think of an API like a restaurant menu.
  - **Menu as API**: The menu provides a list of dishes you can order, along with a description of each dish.
  - **Order Process**: When you decide what you want, you place your order with the waiter (the API).
  - **Kitchen as Service Provider**: The kitchen (the server or application) prepares your food (data or functionality) based on your order.
  - **Delivery**: The waiter then brings your meal back to you, just as an API returns the requested data or service to the user.

### Practical Example from Web Development

- **Weather Data API**: 
  - **Functionality**: A web application that shows weather forecasts can use a weather data API.
  - **How It Works**: 
    - The application sends a request to the weather API asking for the current weather in a specific location.
    - The API processes this request and retrieves the relevant weather data from its database.
    - It then sends this data back to the application, which displays it on the user's screen.
  - **Benefits**: This allows the web application to provide users with up-to-date weather information without having to build its own weather database from scratch.

### Summary

- An API simplifies interactions between different software systems, much like a menu simplifies the ordering process in a restaurant. It enables applications to access and utilize each other's functionalities efficiently, enhancing user experience and saving development time.

### **Judge summary:** Clear, well-structured API explanation with strong analogy but minor format deviation

---

# 🏆 Winner: Prompt C


### 👆 Βέβαια κατα τύχη πήραμε το prompt C ως νικητή απλά επειδή ήταν το τελευταίο max, σύμφωνα με την βαθμολογία και το prompt B θα μπορούσε να είναι νικητής γιατί είχε ισοβαθμία σε όλα με αυτά του C, οπότε και δεν έβαλα άλλο κριτίριο για τον νικητή στην προκειμένη περίπτωση.  